# Umsatzprognose

Dieses Notebook zeigt, wie viel Umsatz aus den Projekten zu erwarten ist, die bereits
in Clockodo angelegt sind. Es liest die Daten nur; es verändert in Clockodo nichts.

**So wird es benutzt:** oben im Menü *Laufzeit → Alle ausführen*, dann von oben nach
unten lesen. Der Abruf dauert etwa eine halbe Minute. Alle Diagramme sind interaktiv –
mit dem Mauszeiger über einem Balken stehen die genauen Zahlen.

In [ ]:
# @title Umgebung einrichten und Dashboard laden

import datetime
import importlib.util

if (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
):
    PAKET_REF = "main"
    PAKET_URL = f"git+https://github.com/it-agile/umsatzprognose-clockodo.git@{PAKET_REF}"
    SETUP_URL = (
        "https://raw.githubusercontent.com/it-agile/umsatzprognose-clockodo/"
        f"{PAKET_REF}/notebooks/setup.py"
    )
    !pip install --quiet "$PAKET_URL"
    !pip install --quiet --force-reinstall --no-deps "$PAKET_URL"
    !curl -sL "$SETUP_URL" -o setup.py


import setup

stichtag = datetime.datetime.now(tz=datetime.UTC).date()
horizont_monate = 3
auslastung_monate = 12
projekte_ohne_auftragsvolumen = 20

dashboard = setup.dashboard(
    stichtag=stichtag,
    horizont_monate=horizont_monate,
    auslastung_monate=auslastung_monate,
)

print(dashboard.ladebericht())

## Parameter und Variablen für die Verarbeitung

In [ ]:
# @title Stundensatz-Korrekturen (optional)

from decimal import Decimal

# Projektname wie in der Hinweistabelle oben, Satz in Euro je Stunde. Beispiel:
# stundensatz_korrekturen = {"Beispielprojekt": Decimal("95.0")}
stundensatz_korrekturen: dict[str, Decimal] = {}

dashboard.stundensatz_uebersteuern(stundensatz_korrekturen)

Zur Kontrolle: der Hinweis "Stundensatz 0" verschwindet für die eingetragenen Projekte.

In [ ]:
# @title Verbrauchsplan-Übersteuerung (optional)

# Projektname wie in der Hinweistabelle, Zielmonat als (Jahr, Monat). Beispiel:
# verbrauchsplan_uebersteuerungen = {"Beispielprojekt": (2026, 12)}
verbrauchsplan_uebersteuerungen: dict[str, tuple[int, int]] = {}

dashboard.verbrauchsplan_uebersteuern(verbrauchsplan_uebersteuerungen)

In [ ]:
# @title Interne Arbeit in der Simulation (optional)

from umsatzprognose.domaene import GaussFakturierbareArbeit, WeibullFakturierbareArbeit

# anteil_fakturierbar: None (Standard) verwendet den historischen Durchschnitt als
# festen, über alle Läufe gleichen Anteil fakturierbarer Arbeit (Modus "Pauschal" in
# der Webapp). Trag hier stattdessen eine feste Zahl zwischen 0.0 und 1.0 ein, um einen
# eigenen Pauschalwert zu erzwingen.
anteil_fakturierbar: float | None = None

# fakturierbare_arbeit_ziehung: alternativ eine parametrische Verteilung, je Lauf,
# Horizontmonat und Person unabhängig gezogen (schließt sich mit anteil_fakturierbar
# aus) - jeweils aus der Historie geschätzt (per Hand änderbar) oder mit eigenen
# Parametern:
#
#   fakturierbare_arbeit_ziehung = WeibullFakturierbareArbeit.aus_stichprobe(
#       dashboard.fakturierbare_arbeit_verteilung().werte
#   )
#   fakturierbare_arbeit_ziehung = WeibullFakturierbareArbeit(
#       formparameter=2.0, skalenparameter=0.8
#   )
#
#   fakturierbare_arbeit_ziehung = GaussFakturierbareArbeit.aus_stichprobe(
#       dashboard.fakturierbare_arbeit_verteilung().werte
#   )
#   fakturierbare_arbeit_ziehung = GaussFakturierbareArbeit(
#       mittelwert=0.85, standardabweichung=0.05
#   )
fakturierbare_arbeit_ziehung: WeibullFakturierbareArbeit | GaussFakturierbareArbeit | None = None

In [ ]:
# @title Simulation ausführen

setup.simulieren(
    dashboard,
    monate=horizont_monate,
    anteil_fakturierbar=anteil_fakturierbar,
    fakturierbare_arbeit_ziehung=fakturierbare_arbeit_ziehung,
)

## Überblick

Links der tatsächlich erzielte Umsatz der vergangenen zwölf abgeschlossenen Monate,
rechts das Volumen, das aus laufenden Projekten noch abgerufen werden kann.

In [ ]:
# @title Kennzahlen im Überblick

dashboard.kennzahlen()

## Umsatz, Kosten und Gewinn je Monat

Alle Buchungen des jeweiligen Monats, auch die ohne Projektbezug. Der letzte Balken
der Historie ist der laufende Monat.

Rechts daran schließt sich der Prognosehorizont an: 
* **bereits gebuchter** Umsatz je Monat
* **prognostizierter** Umsatz obendrauf in gedämpfter Farbe
* **Schulungsanmeldungen**, sofern unten geladen: der schon feststehende Umsatz
  bereits geplanter öffentlicher Schulungstermine, additiv und ohne eigene Bandbreite.

Der dünne Balken am oberen Rand zeigt, wie weit die vorsichtigeren 85-%- und 95-%-Schätzungen darunter liegen.

Zusätzlich, sofern eine Kostenprognose geladen ist, läuft eine **Kosten**-Linie über
die gesamte Breite - Historie und Prognosehorizont gleichermaßen, denn anders als der
Umsatz gilt die Kostenprognose auch für bereits vergangene Monate. Auch sie hat keine
eigene Bandbreite: der Wert steht in der externen Kostenplanung schon fest. Die Lücke
zwischen Umsatzbalken und Kostenlinie ist der **Gewinn**; in der Tabelle darunter steht
er als eigene Spalte, zusammen mit den Kosten.

Der Umsatz bereits geplanter öffentlicher Schulungstermine sowie die Kostenprognose kommen aus derselben separaten Google-Sheets-Tabelle (unterschiedliche Tabellenblätter je Baustein) - ein Login deckt beides ab. In Colab meldest du dich dafür mit deinem eigenen Google-Konto an (ein Login-Fenster öffnet sich); lokal öffnet der erste Aufruf einmalig einen Browser-Tab zum Anmelden, danach läuft es automatisch. Ohne eingetragene Zugangsdaten (siehe `.env.sample` bzw. die passenden Colab-Secrets) bricht die folgende Zelle mit einer Fehlermeldung ab.

In [ ]:
# @title Umsatz, Kosten und Gewinn je Monat

dashboard.umsatzverlauf()

Dieselben Zahlen zum Nachlesen:

In [ ]:
# @title Zahlen zum Nachlesen

dashboard.umsatztabelle()

## Anteil fakturierbarer Arbeit

Minimum, Durchschnitt und Maximum je Monat über alle Personen mit gebuchter Zeit
(Zeit ohne Kundenbezug) - reine Vergangenheitsbetrachtung über die abgeschlossenen
Monate des geladenen Auslastungsfensters. Die Simulation oben zieht standardmäßig aus
dem historischen Durchschnitt als Pauschalwert; der Abschnitt weiter oben lässt dich
stattdessen einen eigenen Pauschalwert oder eine Weibull-/Gauss-Verteilung wählen.

In [ ]:
# @title Anteil fakturierbarer Arbeit je Monat

dashboard.anteil_fakturierbarer_arbeit()

Dieselben Zahlen zum Nachlesen:

In [ ]:
# @title Zahlen zum Nachlesen

dashboard.anteil_fakturierbarer_arbeit_tabelle()

Wie sich die einzelnen Personen-Monate hinter Durchschnitt und Bandbreite oben verteilen, als Histogramm über alle Personen und Monate des geladenen Fensters zusammen: 

In [ ]:
# @title Verteilung Anteil fakturierbarer Arbeit

# Wertebereich einschränken (0.0 bis 1.0), um Ausreißer am unteren bzw. oberen Ende
# gezielt aus der Anzeige auszublenden - ohne Auswirkung auf die Simulation.
anteil_fakturierbarer_arbeit_verteilung_minimum = 0.0
anteil_fakturierbarer_arbeit_verteilung_maximum = 1.0

dashboard.anteil_fakturierbarer_arbeit_verteilung(
    minimum=anteil_fakturierbarer_arbeit_verteilung_minimum,
    maximum=anteil_fakturierbarer_arbeit_verteilung_maximum,
)

## Offenes Auftragsvolumen

Je Projekt der Teil des beauftragten Volumens, der noch nicht verbraucht ist.

In [ ]:
# @title Offenes Auftragsvolumen je Projekt

dashboard.restvolumen_je_projekt(top=projekte_ohne_auftragsvolumen)

## Was zu den Zahlen zu wissen ist

Nicht jedes Projekt lässt sich auswerten. Hier steht, welche Fälle aufgetreten sind
und was sie für die Zahlen oben bedeuten.

In [ ]:
# @title Hinweise zu den Zahlen

dashboard.hinweise(max_anzahl_betroffen=10)

## Projekte ohne Budget
 Ohne gefilterte Projekte, sortiert.

In [ ]:
# @title Projekte ohne Budget

# filter kann hier direkt angepasst werden
projekt_filter = [
    "it-agile GmbH",
    "Öffentliche Schulung",
]

dashboard.projekte_ohne_budget(projekt_filter=projekt_filter)